<a href="https://colab.research.google.com/github/Raksh1707/Naturalproject/blob/main/nlp_046.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

uploaded = files.upload()

Saving activity_botscore.csv to activity_botscore.csv


In [2]:
!pip install -q streamlit pyngrok pandas numpy scikit-learn matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 5.4 MB/s eta 0:00:00


In [3]:
%%writefile app.py

# ============================================================
# SOCIAL MEDIA BOT ACTIVITY DETECTION
# STREAMLIT APPLICATION
# ============================================================

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="Social Media Bot Detection",
    page_icon="🤖",
    layout="wide"
)


# ============================================================
# TITLE
# ============================================================

st.title("🤖 Social Media Bot Activity Detection")

st.write(
    "This application analyzes social-media account behavior "
    "and predicts whether an account is likely to be a Bot or Human."
)

st.info(
    "Dataset features: user_id, age, count, activity, "
    "bot_score_english"
)


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.title("⚙️ Model Settings")

threshold = st.sidebar.slider(
    "Bot Score Threshold",
    min_value=0.10,
    max_value=0.90,
    value=0.50,
    step=0.05
)

test_size = st.sidebar.slider(
    "Testing Dataset Size",
    min_value=0.10,
    max_value=0.40,
    value=0.20,
    step=0.05
)

n_estimators = st.sidebar.slider(
    "Number of Trees",
    min_value=50,
    max_value=300,
    value=100,
    step=50
)


# ============================================================
# FILE UPLOAD
# ============================================================

st.sidebar.header("📂 Upload Dataset")

uploaded_file = st.sidebar.file_uploader(
    "Upload activity_botscore.csv",
    type=["csv"]
)


# ============================================================
# FUNCTION TO LOAD DATA
# ============================================================

@st.cache_data
def load_data(file):

    data = pd.read_csv(file)

    return data


# ============================================================
# MAIN APPLICATION
# ============================================================

if uploaded_file is None:

    st.warning(
        "Please upload your activity_botscore.csv file "
        "from the sidebar."
    )

    st.stop()


# ============================================================
# LOAD DATA
# ============================================================

df = load_data(uploaded_file)


# ============================================================
# CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "user_id",
    "age",
    "count",
    "activity",
    "bot_score_english"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]


if len(missing_columns) > 0:

    st.error(
        "The following required columns are missing:"
    )

    st.write(missing_columns)

    st.write(
        "Expected columns:"
    )

    st.write(required_columns)

    st.stop()


# ============================================================
# DATA PREPROCESSING
# ============================================================

df = df.drop_duplicates()

numeric_columns = [
    "age",
    "count",
    "activity",
    "bot_score_english"
]

for column in numeric_columns:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


df = df.dropna(
    subset=numeric_columns
)


# ============================================================
# CREATE TARGET LABEL
# ============================================================

df["Bot_Label"] = np.where(
    df["bot_score_english"] >= threshold,
    1,
    0
)

df["Bot_Type"] = np.where(
    df["Bot_Label"] == 1,
    "Bot",
    "Human"
)


# ============================================================
# HEADER METRICS
# ============================================================

st.header("📊 Dataset Overview")

col1, col2, col3, col4 = st.columns(4)

with col1:

    st.metric(
        "Total Users",
        len(df)
    )

with col2:

    st.metric(
        "Bot Accounts",
        int((df["Bot_Label"] == 1).sum())
    )

with col3:

    st.metric(
        "Human Accounts",
        int((df["Bot_Label"] == 0).sum())
    )

with col4:

    st.metric(
        "Average Activity",
        round(df["activity"].mean(), 2)
    )


# ============================================================
# DATASET PREVIEW
# ============================================================

st.subheader("📋 Dataset Preview")

st.dataframe(
    df.head(100),
    use_container_width=True
)


# ============================================================
# DATASET INFORMATION
# ============================================================

st.subheader("🔍 Dataset Information")

info_col1, info_col2 = st.columns(2)

with info_col1:

    st.write("Shape:")

    st.write(
        f"{df.shape[0]} rows × {df.shape[1]} columns"
    )

with info_col2:

    st.write("Missing Values:")

    st.write(
        int(df.isnull().sum().sum())
    )


# ============================================================
# BOT / HUMAN DISTRIBUTION
# ============================================================

st.header("👥 Bot vs Human Analysis")

distribution = df["Bot_Type"].value_counts()


col1, col2 = st.columns(2)


with col1:

    fig, ax = plt.subplots(
        figsize=(7, 5)
    )

    ax.bar(
        distribution.index,
        distribution.values
    )

    ax.set_title(
        "Bot vs Human Accounts"
    )

    ax.set_xlabel(
        "Account Type"
    )

    ax.set_ylabel(
        "Number of Accounts"
    )

    st.pyplot(fig)


with col2:

    fig, ax = plt.subplots(
        figsize=(7, 5)
    )

    ax.pie(
        distribution.values,
        labels=distribution.index,
        autopct="%1.1f%%",
        startangle=90
    )

    ax.set_title(
        "Account Distribution"
    )

    st.pyplot(fig)


# ============================================================
# ACCOUNT BEHAVIOR ANALYSIS
# ============================================================

st.header("📈 Account Behavior Analysis")

behavior = df.groupby(
    "Bot_Type"
)[
    [
        "age",
        "count",
        "activity",
        "bot_score_english"
    ]
].mean()

st.dataframe(
    behavior.round(2),
    use_container_width=True
)


# ============================================================
# BOT SCORE DISTRIBUTION
# ============================================================

st.subheader("Bot Score Distribution")

fig, ax = plt.subplots(
    figsize=(10, 5)
)

ax.hist(
    df["bot_score_english"],
    bins=30
)

ax.axvline(
    threshold,
    linestyle="--",
    label=f"Threshold = {threshold}"
)

ax.set_title(
    "Distribution of Bot Scores"
)

ax.set_xlabel(
    "Bot Score"
)

ax.set_ylabel(
    "Number of Users"
)

ax.legend()

st.pyplot(fig)


# ============================================================
# ACTIVITY VS BOT SCORE
# ============================================================

st.subheader(
    "Account Activity vs Bot Score"
)

fig, ax = plt.subplots(
    figsize=(10, 5)
)

sns.scatterplot(
    data=df.sample(
        min(10000, len(df)),
        random_state=42
    ),
    x="activity",
    y="bot_score_english",
    hue="Bot_Type",
    ax=ax
)

ax.set_title(
    "Activity vs Bot Score"
)

ax.set_xlabel(
    "Average Posts Per Day"
)

ax.set_ylabel(
    "Bot Score"
)

st.pyplot(fig)


# ============================================================
# MACHINE LEARNING MODEL
# ============================================================

st.header("🤖 Machine Learning Model")

st.write(
    "Model: Random Forest Classifier"
)

st.write(
    "Features used: age, count, activity"
)

st.write(
    "Target: Bot / Human"
)


# ============================================================
# PREPARE FEATURES
# ============================================================

X = df[
    [
        "age",
        "count",
        "activity"
    ]
]

y = df["Bot_Label"]


# ============================================================
# CHECK BOTH CLASSES
# ============================================================

if y.nunique() < 2:

    st.error(
        "The selected threshold creates only one class. "
        "Please change the Bot Score Threshold."
    )

    st.stop()


# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=test_size,

    random_state=42,

    stratify=y
)


# ============================================================
# RANDOM FOREST
# ============================================================

model = RandomForestClassifier(

    n_estimators=n_estimators,

    random_state=42,

    class_weight="balanced",

    n_jobs=-1
)


model.fit(
    X_train,
    y_train
)


# ============================================================
# PREDICTION
# ============================================================

y_pred = model.predict(
    X_test
)


# ============================================================
# MODEL METRICS
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)


st.subheader("📊 Model Performance")

col1, col2, col3, col4 = st.columns(4)

with col1:

    st.metric(
        "Accuracy",
        f"{accuracy * 100:.2f}%"
    )

with col2:

    st.metric(
        "Precision",
        f"{precision * 100:.2f}%"
    )

with col3:

    st.metric(
        "Recall",
        f"{recall * 100:.2f}%"
    )

with col4:

    st.metric(
        "F1 Score",
        f"{f1 * 100:.2f}%"
    )


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

st.subheader(
    "📑 Classification Report"
)

report = classification_report(
    y_test,
    y_pred,
    target_names=[
        "Human",
        "Bot"
    ],
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(
    report
).transpose()

st.dataframe(
    report_df.round(3),
    use_container_width=True
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

st.subheader(
    "🔲 Confusion Matrix"
)

cm = confusion_matrix(
    y_test,
    y_pred
)

fig, ax = plt.subplots(
    figsize=(7, 5)
)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[
        "Human",
        "Bot"
    ],
    yticklabels=[
        "Human",
        "Bot"
    ],
    ax=ax
)

ax.set_xlabel(
    "Predicted"
)

ax.set_ylabel(
    "Actual"
)

ax.set_title(
    "Bot Detection Confusion Matrix"
)

st.pyplot(fig)


# ============================================================
# FEATURE IMPORTANCE
# ============================================================

st.subheader(
    "⭐ Feature Importance"
)

importance = pd.DataFrame({

    "Feature": [
        "age",
        "count",
        "activity"
    ],

    "Importance": model.feature_importances_

})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

st.dataframe(
    importance.round(4),
    use_container_width=True
)


fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.bar(
    importance["Feature"],
    importance["Importance"]
)

ax.set_title(
    "Feature Importance"
)

ax.set_xlabel(
    "Feature"
)

ax.set_ylabel(
    "Importance"
)

st.pyplot(fig)


# ============================================================
# SUSPICIOUS ACCOUNT ANALYSIS
# ============================================================

st.header(
    "🚨 Suspicious Account Analysis"
)

suspicious = df[
    df["bot_score_english"] >= 0.70
].sort_values(
    by="bot_score_english",
    ascending=False
)

st.write(
    f"Accounts with bot score ≥ 0.70: {len(suspicious)}"
)

st.dataframe(
    suspicious[
        [
            "user_id",
            "age",
            "count",
            "activity",
            "bot_score_english",
            "Bot_Type"
        ]
    ].head(100),
    use_container_width=True
)


# ============================================================
# NEW ACCOUNT PREDICTION
# ============================================================

st.header(
    "🔮 Predict a New Social Media Account"
)

st.write(
    "Enter account behavior below and click Predict."
)


col1, col2, col3 = st.columns(3)


with col1:

    new_age = st.number_input(
        "Account Age (days)",
        min_value=1.0,
        value=120.0
    )


with col2:

    new_count = st.number_input(
        "Total Posts",
        min_value=0.0,
        value=3000.0
    )


with col3:

    new_activity = st.number_input(
        "Average Posts Per Day",
        min_value=0.0,
        value=25.0
    )


predict_button = st.button(
    "🔍 Predict Account"
)


if predict_button:

    new_data = pd.DataFrame({

        "age": [
            new_age
        ],

        "count": [
            new_count
        ],

        "activity": [
            new_activity
        ]

    })


    prediction = model.predict(
        new_data
    )

    probabilities = model.predict_proba(
        new_data
    )


    bot_probability = (
        probabilities[0][1] * 100
    )


    human_probability = (
        probabilities[0][0] * 100
    )


    if prediction[0] == 1:

        st.error(
            "🤖 Prediction: BOT ACCOUNT"
        )

    else:

        st.success(
            "👤 Prediction: HUMAN ACCOUNT"
        )


    col1, col2 = st.columns(2)


    with col1:

        st.metric(
            "Bot Probability",
            f"{bot_probability:.2f}%"
        )


    with col2:

        st.metric(
            "Human Probability",
            f"{human_probability:.2f}%"
        )


# ============================================================
# FINAL ANALYTICAL REPORT
# ============================================================

st.header(
    "📄 Analytical Report"
)

st.write(
    f"""
    **Total Users:** {len(df):,}

    **Bot Accounts:** {(df["Bot_Label"] == 1).sum():,}

    **Human Accounts:** {(df["Bot_Label"] == 0).sum():,}

    **Average Account Age:** {df["age"].mean():.2f} days

    **Average Total Posts:** {df["count"].mean():.2f}

    **Average Daily Activity:** {df["activity"].mean():.2f} posts/day

    **Average Bot Score:** {df["bot_score_english"].mean():.4f}

    **Model Accuracy:** {accuracy * 100:.2f}%

    **Model Precision:** {precision * 100:.2f}%

    **Model Recall:** {recall * 100:.2f}%

    **Model F1 Score:** {f1 * 100:.2f}%
    """
)


# ============================================================
# DOWNLOAD PROCESSED DATA
# ============================================================

st.header(
    "⬇️ Download Results"
)

csv_data = df.to_csv(
    index=False
).encode("utf-8")


st.download_button(
    label="Download Complete Analysis CSV",
    data=csv_data,
    file_name="Bot_Detection_Analysis.csv",
    mime="text/csv"
)


# ============================================================
# FOOTER
# ============================================================

st.markdown("---")

st.caption(
    "Social Media Bot Activity Detection | "
    "Random Forest Machine Learning Model"
)

Writing app.py


In [4]:
!streamlit run app.py --server.port 8501 > /content/streamlit.log 2>&1 &

In [5]:
!cat /content/streamlit.log

In [6]:
!pip install -q pyngrok

In [7]:
from pyngrok import ngrok

In [8]:
ngrok.set_auth_token("3I1isPAAD1Vjo6T8a39Ko6snmLV_7HCTJktzh4mEzpKMSjoPM")

In [9]:
public_url = ngrok.connect(8501)

print("Streamlit URL:")
print(public_url)

Streamlit URL:
NgrokTunnel: "https://scuff-maritime-glandular.ngrok-free.dev" -> "http://localhost:8501"


In [10]:
!cat /content/streamlit.log



2026-08-19 03:10:59.343 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.14.159:8501

